In [21]:
pip install xgboost

In [22]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [23]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    ConfusionMatrixDisplay, classification_report
)

project_root = Path.cwd().parent

proc_dir = project_root / 'data' / 'processed'
model_dir = project_root / 'models'
output_dir =project_root / 'output' / 'evaluation'
os.makedirs(output_dir, exist_ok=True)

class_names   = ["Low", "Moderate", "High"]
feature_cols  = joblib.load(f"{model_dir}/feature_cols2.pkl")
sns.set_theme(style="whitegrid")

In [24]:

X_train = np.load(f"{proc_dir}/X_train2.npy")
X_test  = np.load(f"{proc_dir}/X_test2.npy")
y_train = np.load(f"{proc_dir}/y_train2.npy")
y_test  = np.load(f"{proc_dir}/y_test2.npy")

print(f"X_train : {X_train.shape}")
print(f"X_test : {X_test.shape}")
print(f"y_train dist : {dict(zip(*np.unique(y_train, return_counts=True)))}")

X_train : (800000, 16)
X_test : (200000, 16)
y_train dist : {0: 266666, 1: 266667, 2: 266667}


In [25]:

svm_subsample = 100_000 

MODELS = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=20, min_samples_split=5,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Logistic Regression": LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "SVM": SVC(
        kernel="rbf", C=1.0, class_weight="balanced",
        probability=True, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric="mlogloss", random_state=42, n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1
    ),
}

In [ ]:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scoring = ["accuracy", "f1_macro", "precision_macro", "recall_macro"]
cv_results = {}

SUBSAMPLE_RF  = 200_000
SUBSAMPLE_SVM = 100_000

for name, model in MODELS.items():
    print(f"\nCV -> {name}")
    start = time.time()

    if name == "SVM":
        idx = np.random.RandomState(42).choice(len(X_train), SUBSAMPLE_SVM, replace=False)
        X_cv, y_cv = X_train[idx], y_train[idx]
        print(f"  (subsample {SUBSAMPLE_SVM:,} untuk SVM)")

    elif name == "Random Forest":
        idx = np.random.RandomState(42).choice(len(X_train), SUBSAMPLE_RF, replace=False)
        X_cv, y_cv = X_train[idx], y_train[idx]
        print(f"  (subsample {SUBSAMPLE_RF:,} untuk RF — RAM limitation)")

    else:
        X_cv, y_cv = X_train, y_train

    scores = cross_validate(
        model, X_cv, y_cv,
        cv=skf, scoring=cv_scoring,
        n_jobs=1, return_train_score=False
    )
    elapsed = time.time() - start

    cv_results[name] = {
        "Accuracy" : scores["test_accuracy"].mean(),
        "Precision" : scores["test_precision_macro"].mean(),
        "Recall" : scores["test_recall_macro"].mean(),
        "F1-Score" : scores["test_f1_macro"].mean(),
        "Accuracy +-" : scores["test_accuracy"].std(),
        "F1 +-" : scores["test_f1_macro"].std(),
        "Time (s)" : round(elapsed, 1)
    }
    print(f"Accuracy : {cv_results[name]['Accuracy']:.4f} +- {cv_results[name]['Accuracy +-']:.4f}")
    print(f"F1-macro : {cv_results[name]['F1-Score']:.4f} +- {cv_results[name]['F1 +-']:.4f}")
    print(f"Waktu : {elapsed:.1f}s")


CV -> Random Forest
  (subsample 200,000 untuk RF — RAM limitation)
Accuracy : 0.7501 +- 0.0015
F1-macro : 0.7520 +- 0.0014
Waktu : 81.5s

CV -> Logistic Regression
Accuracy : 0.7558 +- 0.0013
F1-macro : 0.7565 +- 0.0013
Waktu : 20.1s

CV -> SVM
  (subsample 100,000 untuk SVM)


In [ ]:

cv_df = pd.DataFrame(cv_results).T
display_cols = ["Accuracy", "Precision", "Recall", "F1-Score", "Accuracy ±", "F1 ±", "Time (s)"]
cv_df[display_cols].round(4).style \
    .background_gradient(subset=["Accuracy", "F1-Score"], cmap="Greens") \
    .background_gradient(subset=["Time (s)"], cmap="Reds_r")

KeyError: "None of [Index(['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Accuracy ±', 'F1 ±',\n       'Time (s)'],\n      dtype='object')] are in the [columns]"

In [ ]:

subsample_rf  = 200_000  
subsample_svm = 100_000  

trained_models = {}

for name, model in models.items():
    print(f"\nTraining final — {name}")
    start = time.time()

    if name == "SVM":
        idx = np.random.RandomState(42).choice(len(X_train), subsample_svm, replace=False)
        model.fit(X_train[idx], y_train[idx])
        print(f"  (subsample {subsample_svm:,} rows)")

    elif name == "Random Forest":
        idx = np.random.RandomState(42).choice(len(X_train), subsample_rf, replace=False)
        model.fit(X_train[idx], y_train[idx])
        print(f"  (subsample {subsample_rf:,} rows — RAM limitation)")

    else:
        model.fit(X_train, y_train)

    trained_models[name] = model
    print(f"Done in {time.time() - start:.1f}s")


Training final — Random Forest


MemoryError: could not allocate 2097152 bytes

In [ ]:

test_results = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    test_results[name] = {
        "Accuracy"  : accuracy_score(y_test, y_pred),
        "Precision" : precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Recall" : recall_score(y_test, y_pred, average="macro", zero_division=0),
        "F1-Score" : f1_score(y_test, y_pred, average="macro", zero_division=0),
        "ROC-AUC" : roc_auc_score(y_test, y_prob, multi_class="ovr", average="macro"),
    }
    print(f"\n--{name}--")
    print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:

test_df = pd.DataFrame(test_results).T.round(4)
test_df.to_csv(f"{output_dir}/model_comparison.csv")

test_df.style \
    .background_gradient(subset=["Accuracy", "F1-Score", "ROC-AUC"], cmap="Greens") \
    .highlight_max(subset=["Accuracy", "F1-Score", "ROC-AUC"], color="#d4edda")

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=axes[i], colorbar=False, cmap="Blues")
    axes[i].set_title(f"{name}", fontsize=11)

axes[-1].set_visible(False)
plt.suptitle("Confusion Matrix — Semua Model (Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{output_dir}/confusion_matrices.png", dpi=150)
plt.show()

In [ ]:

FI_models = ["Random Forest", "XGBoost", "LightGBM"]
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for i, name in enumerate(FI_models):
    model = trained_models[name]
    fi_df = pd.DataFrame({
        "Feature"    : feature_cols,
        "Importance" : model.feature_importances_
    }).sort_values("Importance", ascending=False).head(12)

    sns.barplot(data=fi_df, x="Importance", y="Feature",
                palette="viridis", ax=axes[i])
    axes[i].set_title(f"Top 12 Features\n{name}", fontsize=11)
    axes[i].set_xlabel("Importance")
    axes[i].set_ylabel("")

plt.suptitle("Feature Importance Comparison", fontsize=13)
plt.tight_layout()
plt.savefig(f"{output_dir}/feature_importance.png", dpi=150)
plt.show()

In [ ]:
roc_scores = {name: res["ROC-AUC"] for name, res in test_results.items()}

plt.figure(figsize=(9, 4))
bars = plt.barh(list(roc_scores.keys()), list(roc_scores.values()),
                color=["#2196F3","#9E9E9E","#FF9800","#4CAF50","#9C27B0"])
plt.axvline(x=0.5, color="red", linestyle="--", linewidth=1, label="Random baseline")
for bar, val in zip(bars, roc_scores.values()):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", va="center", fontsize=10)
plt.xlim(0.4, 1.05)
plt.title("ROC-AUC Score — Semua Model (Test Set)")
plt.xlabel("ROC-AUC (macro OvR)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/roc_auc_comparison.png", dpi=150)
plt.show()